# Atom Training on Google Colab

Bootstrap, preflight, quick infra check, full run, and resume in one notebook.



## 1) Mount Drive


In [ ]:
from google.colab import drive
#drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)

In [ ]:
rm -rf /content/atom


## 2) Configure variables

Defaults are set for your current public repo/branch, but values are only set if missing.



In [ ]:
import os
import shlex
import shutil
import signal
import subprocess
import sys
from pathlib import Path

# Repo defaults (only applied when missing)
os.environ.setdefault('ATOM_REPO_URL', 'https://github.com/MBifolco/atom.git')
os.environ.setdefault('ATOM_BRANCH', 'main')

# Runtime/cache defaults
os.environ.setdefault('ATOM_DRIVE_REPO', '/content/drive/MyDrive/dev/atom')
os.environ.setdefault('ATOM_WORK_REPO', '/content/atom')
os.environ.setdefault('ATOM_INSTALL_JAX_CUDA', '1')
os.environ.setdefault('ATOM_JAX_VERSION', '0.7.2')
os.environ.setdefault('ATOM_DRIVE_REPO_SYNC_MODE', 'stash')  # stash|reset|skip_pull
os.environ.setdefault('ATOM_REQUIREMENTS_FILE', f"{os.environ['ATOM_WORK_REPO']}/requirements-colab.txt")

# Training defaults
os.environ.setdefault('ATOM_USE_VMAP', '1')
os.environ.setdefault('ATOM_SMOKE_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/quick_test')
os.environ.setdefault('ATOM_FULL_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/run1')
os.environ.setdefault('ATOM_TRAINING_SEED', '1337')
os.environ.setdefault('ATOM_RESUME_CHECKPOINT_DIR', os.environ['ATOM_FULL_OUTPUT_DIR'])
os.environ.setdefault('ATOM_RESUME_START_GEN', '8')
os.environ.setdefault('ATOM_RESUME_TOTAL_GENS', '20')

# Auth defaults (public repo workflow)
os.environ.setdefault('ATOM_USE_GITHUB_TOKEN', '0')
if os.environ.get('ATOM_USE_GITHUB_TOKEN') != '1':
    os.environ.pop('GITHUB_TOKEN', None)

# Private repo option:
# import getpass
# os.environ['ATOM_USE_GITHUB_TOKEN'] = '1'
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ').strip()

os.environ['ATOM_BRANCH'] = 'speed'
for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
    'ATOM_DRIVE_REPO_SYNC_MODE',
    'ATOM_REQUIREMENTS_FILE',
    'ATOM_USE_VMAP',
    'ATOM_SMOKE_OUTPUT_DIR',
    'ATOM_FULL_OUTPUT_DIR',
    'ATOM_TRAINING_SEED',
    'ATOM_RESUME_CHECKPOINT_DIR',
    'ATOM_RESUME_START_GEN',
    'ATOM_RESUME_TOTAL_GENS',
    'ATOM_USE_GITHUB_TOKEN',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)

RUNTIME_MONITOR_LOG = Path('/tmp/atom_runtime_monitor.log')
RUNTIME_MONITOR_PID = Path('/tmp/atom_runtime_monitor.pid')


def run_streaming(cmd, *, cwd, label, raise_on_error=True, env=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"[{label}] $ {printable}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
    finally:
        rc = proc.wait()

    print(f"\n{label} exit code: {rc}")
    if raise_on_error and rc != 0:
        raise RuntimeError(f"{label} failed with exit code {rc}")
    return rc


def _capture_output(cmd):
    try:
        result = subprocess.run(
            [str(part) for part in cmd],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        output = result.stdout.strip()
        return output or f"(exit code {result.returncode}, no output)"
    except FileNotFoundError:
        return f"command not found: {cmd[0]}"


def show_runtime_snapshot():
    print('=== RAM ===')
    print(_capture_output(['free', '-h']))
    print('\n=== GPU ===')
    if shutil.which('nvidia-smi'):
        print(_capture_output([
            'nvidia-smi',
            '--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu',
            '--format=csv',
        ]))
    else:
        print('nvidia-smi not available')


def _read_monitor_pid():
    if not RUNTIME_MONITOR_PID.exists():
        return None
    try:
        return int(RUNTIME_MONITOR_PID.read_text().strip())
    except ValueError:
        return None


def stop_runtime_monitor():
    pid = _read_monitor_pid()
    if pid is None:
        print('No runtime monitor is currently running.')
        RUNTIME_MONITOR_PID.unlink(missing_ok=True)
        return

    try:
        os.kill(pid, signal.SIGTERM)
        print(f'Stopped runtime monitor (pid {pid}).')
    except ProcessLookupError:
        print(f'Runtime monitor pid {pid} was not running.')
    finally:
        RUNTIME_MONITOR_PID.unlink(missing_ok=True)


def start_runtime_monitor(interval_seconds=5):
    stop_runtime_monitor()

    monitor_code = r'''
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

interval_seconds = float(sys.argv[1])
log_path = Path(sys.argv[2])


def capture(cmd):
    try:
        result = subprocess.run(
            cmd,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        output = result.stdout.rstrip()
        return output or f"(exit code {result.returncode}, no output)"
    except FileNotFoundError:
        return f"command not found: {cmd[0]}"


with log_path.open('a', buffering=1) as fh:
    fh.write(f"Runtime monitor started at {datetime.now().isoformat()}\n")
    fh.write(f"Interval: {interval_seconds:.1f}s\n")
    fh.write('=' * 80 + '\n')
    while True:
        fh.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]\n")
        fh.write(capture(['free', '-h']) + '\n\n')
        if shutil.which('nvidia-smi'):
            fh.write(capture([
                'nvidia-smi',
                '--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu',
                '--format=csv',
            ]) + '\n')
        else:
            fh.write('nvidia-smi not available\n')
        fh.write('-' * 80 + '\n')
        fh.flush()
        time.sleep(interval_seconds)
'''

    RUNTIME_MONITOR_LOG.write_text('')
    proc = subprocess.Popen(
        [sys.executable, '-u', '-c', monitor_code, str(interval_seconds), str(RUNTIME_MONITOR_LOG)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    RUNTIME_MONITOR_PID.write_text(str(proc.pid))
    print(f'Started runtime monitor (pid {proc.pid}).')
    print(f'Writing to: {RUNTIME_MONITOR_LOG}')


def tail_runtime_monitor(lines=40):
    if not RUNTIME_MONITOR_LOG.exists():
        print(f'No runtime monitor log found at {RUNTIME_MONITOR_LOG}')
        return
    content = RUNTIME_MONITOR_LOG.read_text().splitlines()
    for line in content[-lines:]:
        print(line)


print('Shared helpers ready: run_streaming, show_runtime_snapshot, start_runtime_monitor, tail_runtime_monitor, stop_runtime_monitor')


## 3) Clone (if needed), preflight, and bootstrap

This handles stale partial clones, validates configuration, and then runs `colab_bootstrap.sh`.

The cell now streams output live so you can see clone, sync, rsync, pip, and JAX install progress as it happens.


In [ ]:
import os
import shutil
from pathlib import Path

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
repo_url = os.environ.get("ATOM_REPO_URL", "")
branch = os.environ.get("ATOM_BRANCH", "main")
auth_repo_url = repo_url
sync_mode = os.environ.get("ATOM_DRIVE_REPO_SYNC_MODE", "<unset>")

print(f"Using WORK_REPO={work_repo}")
print(f"Using BRANCH={branch}")
print(f"ATOM_DRIVE_REPO_SYNC_MODE = {sync_mode}")

if not repo_url:
    raise RuntimeError("ATOM_REPO_URL must be set before first clone.")

if "<org>" in repo_url or "<repo>" in repo_url:
    raise RuntimeError("ATOM_REPO_URL still contains placeholders. Set a real repo URL first.")

if (
    os.environ.get("ATOM_USE_GITHUB_TOKEN", "0") == "1"
    and os.environ.get("GITHUB_TOKEN")
    and repo_url.startswith("https://github.com/")
):
    auth_repo_url = repo_url.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@", 1)

work_repo_path = Path(work_repo)
if work_repo_path.exists() and not (work_repo_path / ".git").exists():
    print(f"Found stale non-git directory at {work_repo}. Removing it...")
    shutil.rmtree(work_repo)

if not (work_repo_path / ".git").exists():
    run_streaming(
        ["git", "clone", "--branch", branch, "--single-branch", auth_repo_url, work_repo],
        cwd=work_repo_path.parent,
        label="work repo clone",
        raise_on_error=True,
    )

run_streaming(
    ["python", "-u", "-m", "src.atom.training.utils.colab_preflight", "--stage", "bootstrap", "--strict"],
    cwd=work_repo,
    label="bootstrap preflight",
    raise_on_error=True,
)

bootstrap_env = os.environ.copy()
bootstrap_env["ATOM_SKIP_PREFLIGHT"] = "1"
run_streaming(
    ["bash", "scripts/colab/bootstrap.sh"],
    cwd=work_repo,
    label="colab bootstrap",
    raise_on_error=True,
    env=bootstrap_env,
)



## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.atom.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 4b) Optional runtime monitor

Colab runs one notebook cell at a time, so the live monitor runs as a detached
background process and writes to `/tmp/atom_runtime_monitor.log` inside the
**Colab runtime**, not on your local machine.

If your terminal prompt looks like `biff@...`, you are on your local machine and
will not be able to `tail` the Colab runtime's `/tmp` directory directly.

Suggested workflow:
1. Run the next cell to print a snapshot and start the monitor.
2. Start quick smoke, full training, or resume.
3. Inspect the log either:
   - from a terminal attached to the Colab runtime
   - or from the notebook with `tail_runtime_monitor()`
4. Stop it with `stop_runtime_monitor()` when you are done.


In [ ]:
show_runtime_snapshot()
start_runtime_monitor(interval_seconds=5)
print()
print(f'Monitor log inside Colab runtime: {RUNTIME_MONITOR_LOG}')
print('If your shell prompt is local (for example biff@...), that terminal cannot tail this file directly.')
print('Live tail from a terminal attached to the Colab runtime:')
print(f'  tail -f {RUNTIME_MONITOR_LOG}')
print('Inspect recent samples from the notebook between runs:')
print('  tail_runtime_monitor(lines=60)')
print('Stop it when finished:')
print('  stop_runtime_monitor()')


## 5) Quick infrastructure smoke test (streaming)

This validates wiring and streams logs live. Quick mode may fail graduation by design.



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
smoke_output_dir = os.environ.get("ATOM_SMOKE_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/quick_test")
training_seed = os.environ.get("ATOM_TRAINING_SEED", "1337")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "smoke",
    "--output-dir", smoke_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="smoke preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "quick",
    "--device", "auto",
    "--seed", str(training_seed),
    "--output-dir", smoke_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="quick smoke", raise_on_error=False)


## 6) Full training run (real run, streaming)



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
full_output_dir = os.environ.get("ATOM_FULL_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/run1")
training_seed = os.environ.get("ATOM_TRAINING_SEED", "1337")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "full",
    "--output-dir", full_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="full preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "complete",
    "--device", "auto",
    "--timesteps", "2000000",
    "--population", "8",
    "--generations", "10",
    "--n-parallel-fighters", "8",
    "--seed", str(training_seed),
    "--output-dir", full_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="full training", raise_on_error=True)


## 7) Resume interrupted run (streaming)



In [ ]:
import os

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
checkpoint_dir = os.environ.get("ATOM_RESUME_CHECKPOINT_DIR", "/content/drive/MyDrive/atom_runs/run1")
start_gen = os.environ.get("ATOM_RESUME_START_GEN", "8")
total_gens = os.environ.get("ATOM_RESUME_TOTAL_GENS", "20")
training_seed = os.environ.get("ATOM_TRAINING_SEED", "1337")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"

preflight_cmd = [
    "python", "-u", "-m", "src.atom.training.utils.colab_preflight",
    "--stage", "resume",
    "--checkpoint-dir", checkpoint_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="resume preflight", raise_on_error=True)

resume_cmd = [
    "python", "-u", "scripts/training/resume_population_training.py",
    "--checkpoint-dir", checkpoint_dir,
    "--start-gen", str(start_gen),
    "--total-gens", str(total_gens),
    "--seed", str(training_seed),
]
if use_vmap:
    resume_cmd.append("--use-vmap")

run_streaming(resume_cmd, cwd=work_repo, label="resume run", raise_on_error=True)


In [ ]:
%%bash
set -euo pipefail
python -m pip install -U funkybob


## Troubleshooting

- Public repo: keep `ATOM_USE_GITHUB_TOKEN=0`.
- Private repo: set `ATOM_USE_GITHUB_TOKEN=1` and provide `GITHUB_TOKEN`.
- If Drive cache has local edits, bootstrap auto-stashes by default (`ATOM_DRIVE_REPO_SYNC_MODE=stash`).
- Use `ATOM_DRIVE_REPO_SYNC_MODE=reset` to discard cache edits, or `skip_pull` to avoid pulling when dirty.
- Use `python -m src.atom.training.utils.colab_preflight --stage <bootstrap|smoke|full|resume> --strict` for fast diagnostics.
- Milestone gate checklist: `docs/COLAB_VALIDATION_CHECKLIST.md`.

